In [2]:
import json
with open('data/nuscenes_drive_data_single_image_val_inference.json', 'r') as f:
    infer_data = json.load(f)

In [37]:
import numpy as np
np.array(get_status(infer_data[3][1]))

array([[ 0.77, -0.01],
       [ 0.6 , -0.01],
       [ 0.54, -0.01],
       [ 0.46, -0.02],
       [ 0.38, -0.02],
       [ 0.36, -0.03],
       [ 0.51, -0.  ],
       [ 0.68,  0.01],
       [ 0.76,  0.01],
       [ 0.68,  0.01]])

In [55]:
import re

s = "Future speeds and curvatures: [0.00, 72.36], [0.01, -2.51], [0.14, 0.08], [0.27, 0.01], [0.27, 0.01], [0.63, 0.02], [0.82, 0.01], [0.96, 0.01], [0.96, 0.00], [0.82, -0.01]"
def get_status(s):
    # Find all [x, y] pairs using regex
    raw_string = s.replace("'", "")                   # Remove stray single quotes
    raw_string = re.sub(r"\[+", "[", raw_string)               # Remove extra opening brackets
    raw_string = re.sub(r"\]+", "]", raw_string)               # Remove extra closing brackets
    raw_string = re.sub(r"\s+", "", raw_string)                # Remove all whitespaces
    raw_string = re.sub(r",(?=\])", "", raw_string)            # Remove trailing commas before closing brackets
    raw_string = re.sub(r"\[([0-9])", r"[\1", raw_string)       # Ensure correct bracket-number spacing
    raw_string = re.sub(r"([0-9])\]", r"\1]", raw_string)       # Same for trailing numbers
    raw_string = re.sub(r"\.,", ",", raw_string)               # Remove misused period commas

    matches = re.findall(r'\[([^\]]+)\]', raw_string)
    # Find the maximum length of the sublists
    try:
        data = [list(map(float, match.split(','))) for match in matches]
        max_len = max(len(sublist) for sublist in data)

    # # Pad sublists with 0.0 to make them all the same length
    # padded_data = [sublist + [0.0] * (max_len - len(sublist)) for sublist in data]
    # padded_data = [sublist if len(sublist) == max_len else sublist + [sublist[-1]] * (max_len - len(sublist)) for sublist in data]


    # try:
        return [sublist if len(sublist) == max_len else sublist + [sublist[-1]] * (max_len - len(sublist)) for sublist in data]
    except:
        return []

    # Convert them into a list of lists with floats
    for match in matches:
        try:
            result = [list(map(float, match.split(',')))]
        except:
            continue

    # result = [list(map(float, match.split(','))) for match in matches]
    return result

# print(result)

In [97]:
np.array(get_status(data_sample[0][0]))

array([[ 0.e+00,  1.e+04],
       [ 0.e+00, -5.e+03],
       [ 1.e-03,  0.e+00],
       [ 0.e+00,  0.e+00],
       [ 0.e+00,  0.e+00],
       [ 1.e-03,  0.e+00],
       [ 0.e+00,  0.e+00],
       [ 0.e+00,  5.e+04],
       [ 0.e+00,  5.e+04],
       [ 0.e+00, -1.e+05]])

In [78]:
data_sample[0][0]

'Future speeds and curvatures: [0.00, -499.95], [0.00, -2623.02], [0.00, 5705.97], [0.00, -582.00], [0.00, -10819.07], [0.00, 16122.92], [0.00, -22884.21], [0.00, 2468.90]'

In [43]:

def EstimateCurvatureFromTrajectory(traj):
    traj = traj[:, :2]
    curvature = np.zeros(len(traj))

    for i in range(1, len(traj) - 1):
        x1, y1 = traj[i - 1]
        x2, y2 = traj[i]
        x3, y3 = traj[i + 1]

        # Vectors
        v1 = np.array([x2 - x1, y2 - y1])
        v2 = np.array([x3 - x2, y3 - y2])

        # Lengths
        L1 = np.linalg.norm(v1)
        L2 = np.linalg.norm(v2)
        L3 = np.linalg.norm(np.array([x3 - x1, y3 - y1]))

        # Signed area (using cross product)
        area_signed = 0.5 * ((x2 - x1)*(y3 - y1) - (y2 - y1)*(x3 - x1))

        if L1 > 0 and L2 > 0 and L3 > 0:
            curvature[i] = 4 * area_signed / (L1 * L2 * L3)

    curvature[0] = curvature[1]
    curvature[-1] = curvature[-2]

    return curvature


In [44]:
from scipy.integrate import cumulative_trapezoid
def IntegrateCurvatureForPoints(curvatures, velocities_norm, initial_position, initial_heading, time_span):
    t = np.linspace(0, time_span, time_span)  # Time vector

    # Initial conditions
    x0, y0 = initial_position[0], initial_position[1]  # Starting position
    theta0 = initial_heading  # Initial orientation (radians)

    # Integrate to compute heading (theta)
    theta = cumulative_trapezoid(curvatures * velocities_norm, t, initial=0)
    theta += theta0  # 手动加上初始角度

    # Compute velocity components
    v_x = velocities_norm * np.cos(theta)
    v_y = velocities_norm * np.sin(theta)

    # Integrate to compute trajectory
    x = cumulative_trapezoid(v_x, t, initial=0)
    y = cumulative_trapezoid(v_y, t, initial=0)
    x += x0  # 手动加上初始位置
    y += y0

    return np.stack((x, y), axis=1)

In [ ]:
for data_sample in infer_data:
    pred_ego_states = get_status(data_sample[0][0])
    gt_ego_states = get_status(data_sample[1])
    IntegrateCurvatureForPoints(pred_ego_states[1], pred_ego_states[0], initial_position, initial_heading, time_span)
    break

In [42]:
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN

In [3]:
from nuscenes import NuScenes
nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
scenes = nusc.scene
from nuscenes.utils.splits import create_splits_scenes

print(f"Number of scenes: {len(scenes)}")
val_scenes = create_splits_scenes()['val']
for i,scene in enumerate(scenes):
    data_sample = infer_data[i]
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # # nusc.render_sample_data(cam_front_data['token'])


        # if "gpt" in args.model_path:
        #     with open(os.path.join(nusc.dataroot, cam_front_data['filename']), "rb") as image_file:
        #         front_camera_images.append(base64.b64encode(image_file.read()).decode('utf-8'))
        # else:
        #     front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # Get the camera parameters of the sample.
        camera_params.append(nusc.get('calibrated_sensor', cam_front_data['calibrated_sensor_token']))

        # Advance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    ## Compute interpolated trajectory.
    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    pred_ego_states = get_status(data_sample[0][0])
    gt_ego_states = get_status(data_sample[1])
    pred_points = IntegrateCurvatureForPoints(pred_ego_states[1], pred_ego_states[0], ego_poses_world[0], atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    # estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
    #                                                 atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)

Loading NuScenes tables for version v1.0-trainval...
23 category,
8 attribute,
4 visibility,
64386 instance,
12 sensor,
10200 calibrated_sensor,
2631083 ego_pose,
68 log,
850 scene,
34149 sample,
2631083 sample_data,
1166187 sample_annotation,
4 map,
Done loading in 16.740 seconds.
Reverse indexing ...
Done reverse indexing in 3.2 seconds.
Number of scenes: 850
Scene scene-0003 has 0 frames
Scene scene-0003 has less than 20 frames, skipping...
Scene scene-0012 has 0 frames
Scene scene-0012 has less than 20 frames, skipping...
Scene scene-0013 has 0 frames
Scene scene-0013 has less than 20 frames, skipping...
Scene scene-0014 has 0 frames
Scene scene-0014 has less than 20 frames, skipping...
Scene scene-0015 has 0 frames
Scene scene-0015 has less than 20 frames, skipping...
Scene scene-0016 has 0 frames
Scene scene-0016 has less than 20 frames, skipping...
Scene scene-0017 has 0 frames
Scene scene-0017 has less than 20 frames, skipping...
Scene scene-0018 has 0 frames
Scene scene-0018 h

In [4]:
def convert_to_speed_curvature_template(speed_array, curvature_array):
    """
    Convert speed array and curvature array to template string format.
    Returns: '[speed_1, curvature_1], [speed_2, curvature_2],..., [speed_10, curvature_10]'
    """
    # Create the template pairs
    pairs = []
    for i in range(len(curvature_array)):
        pair_str = f"[{speed_array[i]:.2f}, {curvature_array[i]:.2f}]"
        pairs.append(pair_str)
    
    # Join all pairs with commas
    result = ", ".join(pairs)
    return result
def convert_with_format(num):
    return "{:012d}".format(num)

In [5]:
from nuscenes import NuScenes
nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
scenes = nusc.scene

Loading NuScenes tables for version v1.0-trainval...
23 category,
8 attribute,
4 visibility,
64386 instance,
12 sensor,
10200 calibrated_sensor,
2631083 ego_pose,
68 log,
850 scene,
34149 sample,
2631083 sample_data,
1166187 sample_annotation,
4 map,
Done loading in 15.752 seconds.
Reverse indexing ...
Done reverse indexing in 3.2 seconds.


In [207]:
import json
with open('/home/can/Desktop/research/LLaDA-V/data_062525/results/nuscenes_drive_data_single_image_val_inference_zero_shot.json', 'r') as f:
    infer_data = json.load(f)

In [258]:
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes


val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid = 0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # Get the camera parameters of the sample.
        camera_params.append(nusc.get('calibrated_sensor', cam_front_data['calibrated_sensor_token']))

        # Advance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    ## Compute interpolated trajectory.
    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)

    # Debug
    # if args.plot:
    #     plt.quiver(ego_poses_world[:, 0], ego_poses_world[:, 1], ego_velocities[:, 0], ego_velocities[:, 1],
    #             color='b')
    #     plt.plot(estimated_points[:, 0], estimated_points[:, 1], 'g-', label='Reconstruction')
    #     plt.legend()
    #     plt.savefig(f"{timestamp}/{name}_interpolation.jpg")
    #     plt.close()



    # Get the waypoints of the ego vehicle.
    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []
    # ade1s_list = []
    # ade2s_list = []
    # ade3s_list = []

# {
#   "image": ["images/cat1.jpg", "images/cat2.jpg", "images/cat3.jpg"],
#   "conversations": [{"from": "human", "value": "<image>\n<image>\n<image>\nCompare these cats."}]
# }

    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # idx+=1

        data_sample = infer_data[idx]
        idx += 1
        # if idx == 3000:
        #     break

        
        speed_curvature_pred = get_status(data_sample[0][0])
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid+=1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            speed_curvature_pred = speed_curvature_pred[:pred_len]
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            invalid+=1
            continue
        fde1s_list.append(fde1s)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        fde2s_list.append(fde2s)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        fde3s_list.append(fde3s)

        fut_ego_traj_world = np.array(fut_ego_traj_world)
        # ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        # pred1_len = min(pred_len, 2)
        # ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        # ade1s_list.append(ade1s)

        # pred2_len = min(pred_len, 4)
        # ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        # ade2s_list.append(ade2s)

        # pred3_len = min(pred_len, 6)
        # ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # # print(ade3s_list)
        # ade3s_list.append(ade3s)

        # # Compute FDE.
        # fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # # print(f"ade: {ade}, fde: {fde}")
        # fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        # fde1s_list.append(fde1s)
        # fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        # fde2s_list.append(fde2s)
        # fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[1:pred3_len+1][-1])
        # fde3from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LENist = [x for x in fde2s_list if isinstance(x, (int, float)) and not math.isnan(x)]

SyntaxError: cannot assign to expression (2206123382.py, line 204)

In [83]:
np.mean(ade1s_list), np.mean(ade2s_list), np.mean(ade3s_list)

(0.8705697738019549, 1.5928235717057762, 3.184141482246658)

In [222]:
np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list)

(1.0949088434217102, nan, nan)

In [223]:
np.mean(valid_fde2s_list), np.mean(valid_fde3s_list)

(2.523424544465264, 3.763810861063727)

In [209]:
np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list)

(1.1819740589110015, 2.7230883614871138, 4.048548138657476)

In [75]:
mean_fde = np.mean(fde1s_list + fde2s_list + fde3s_list)
print(f"Mean FDE: {mean_fde:.4f}")

Mean FDE: 2.6520


In [224]:
invalid

33

## ZS + COT

In [218]:
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes


val_scenes = create_splits_scenes()['val']
import json
with open('/home/can/Desktop/research/LLaDA-V/data_062525/results/nuscenes_drive_data_single_image_val_inference_zero_shot_cot.json', 'r') as f:
    infer_data = json.load(f)

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
idx = 0 
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # Get the camera parameters of the sample.
        camera_params.append(nusc.get('calibrated_sensor', cam_front_data['calibrated_sensor_token']))

        # Advance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    ## Compute interpolated trajectory.
    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)

    # Debug
    # if args.plot:
    #     plt.quiver(ego_poses_world[:, 0], ego_poses_world[:, 1], ego_velocities[:, 0], ego_velocities[:, 1],
    #             color='b')
    #     plt.plot(estimated_points[:, 0], estimated_points[:, 1], 'g-', label='Reconstruction')
    #     plt.legend()
    #     plt.savefig(f"{timestamp}/{name}_interpolation.jpg")
    #     plt.close()



    # Get the waypoints of the ego vehicle.
    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []
    # ade1s_list = []
    # ade2s_list = []
    # ade3s_list = []

# {
#   "image": ["images/cat1.jpg", "images/cat2.jpg", "images/cat3.jpg"],
#   "conversations": [{"from": "human", "value": "<image>\n<image>\n<image>\nCompare these cats."}]
# }

    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # idx+=1

        data_sample = infer_data[idx]
        idx += 1
        # if idx == 3000:
        #     break

        
        speed_curvature_pred = get_status(data_sample[0][0])
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            print(fde1s)
            continue
        fde1s_list.append(fde1s)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        fde2s_list.append(fde2s)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        fde3s_list.append(fde3s)

        # import math
        # if not (isinstance(fde3s, (int, float)) and not math.isnan(fde3s)):
        #     print(f"Invalid FDE3s: {fde3s}, skipping this sample.")
        #     print(fut_ego_traj_world[:pred3_len][-1])
        #     print(pred_speeds)
        #     print(pred_curvatures)
        #     break

        # idx+=1

        fut_ego_traj_world = np.array(fut_ego_traj_world)
        ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        pred1_len = min(pred_len, 2)
        ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        ade1s_list.append(ade1s)

        pred2_len = min(pred_len, 4)
        ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        ade2s_list.append(ade2s)

        pred3_len = min(pred_len, 6)
        ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # print(ade3s_list)
        ade3s_list.append(ade3s)

        # # Compute FDE.
        # fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # # print(f"ade: {ade}, fde: {fde}")
        # fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        # fde1s_list.append(fde1s)
        # fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        # fde2s_list.append(fde2s)
        # fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[1:pred3_len+1][-1])
        # fde3s_list.append(fde3s)

Number of scenes: 150


In [209]:
len(fde3s_list)

2635

In [205]:
fde3s_list[-2]

19.249882372601302

In [203]:
def calculate_mean(numbers):
    return sum(numbers) / len(numbers)
calculate_mean(fde3s_list)

nan

In [175]:
len(ade3s_list), len(fde3s_list)

(2654, 2654)

In [184]:
infer_data[860]

[['Future speeds and curvatures:\n[0.35, 0.19], [0.57, 0.16], [0.80, 0.07], [1.05, -0.01], [1.30, -0.03], [1.61, -0.04], [1.66, -0.04], [1.71, -0.05], [1.80, -0.05], [1.93, -0.05]'],
 '[2.05, -0.05], [2.17, -0.04], [2.31, -0.03], [2.37, -0.02], [2.43, -0.01], [2.40, -0.01], [2.42, -0.02], [2.30, -0.02], [2.27, -0.02], [2.24, -0.02]']

In [178]:
import math
valid_numbers = [x for x in fde3s_list if isinstance(x, (int, float)) and not math.isnan(x)]

In [183]:
for i, x in enumerate(fde3s_list):
    if isinstance(x, (int, float)) and not math.isnan(x):
        continue
    else:
        print(f"Invalid number found: {i,x}")

Invalid number found: (860, nan)


In [180]:
np.mean(valid_numbers)

4.474891078719038

In [168]:
np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list)

(1.3557923795852438, 2.98359184046871, nan)

# LORA

In [141]:
## train shortcut
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes

import json
with open('/home/can/Desktop/research/LLaDA-V/data_062525/results/nuscenes_drive_data_single_image_val_inference_lora_single_image_train_lora.json', 'r') as f:
    infer_data = json.load(f)

val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid = 0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # nuscenes_drive_data_single_image_val_inference_lora_single_image_train_loraAdvance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []


    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # idx+=1

        data_sample = infer_data[idx]
        idx += 1
        # if idx == 3000:
        #     break

        
        speed_curvature_pred = get_status(data_sample[0][0])
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid+=1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            speed_curvature_pred = speed_curvature_pred[:pred_len]
            # continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            continue
        fde1s_list.append(fde1s)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        fde2s_list.append(fde2s)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        fde3s_list.append(fde3s)

        fut_ego_traj_world = np.array(fut_ego_traj_world)
        ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        pred1_len = min(pred_len, 2)
        ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        ade1s_list.append(ade1s)

        pred2_len = min(pred_len, 4)
        ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        ade2s_list.append(ade2s)

        pred3_len = min(pred_len, 6)
        ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # print(ade3s_list)
        ade3s_list.append(ade3s)

        # # Compute FDE.
        # fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # # print(f"ade: {ade}, fde: {fde}")
        # fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        # fde1s_list.append(fde1s)
        # fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        # fde2s_list.append(fde2s)
        # fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[1:pred3_len+1][-1])
        # fde3s_list.append(fde3s)

Number of scenes: 150


In [142]:
np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list)

(0.7981784291554217, 1.9231314924218401, 2.832336901290355)

In [143]:
invalid

18

In [144]:
# np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list)

In [163]:
len(infer_data)

3019

## Shortcut

In [147]:
## train shortcut
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes

import json
with open('/home/can/Desktop/research/LLaDA-V/data_062525/results/nuscenes_drive_data_single_image_val_inference_sc_single_image_train_lora_sc.json', 'r') as f:
    infer_data = json.load(f)

val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid = 0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # Get the camera parameters of the sample.
        camera_params.append(nusc.get('calibrated_sensor', cam_front_data['calibrated_sensor_token']))

        # Advance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []


    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # idx+=1

        data_sample = infer_data[idx]
        idx += 1
        # if idx == 3000:
        #     break

        
        speed_curvature_pred = get_status(data_sample[0][0])
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid +=1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            speed_curvature_pred = speed_curvature_pred[:pred_len]
            # continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            continue
        fde1s_list.append(fde1s)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        fde2s_list.append(fde2s)
        valid_fde2s_list = [fde for fde in fde2s_list if isinstance(fde, (int, float)) and not np.isnan(fde)]
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        fde3s_list.append(fde3s)
        valid_fde3s_list = [fde for fde in fde3s_list if isinstance(fde, (int, float)) and not np.isnan(fde)]

        fut_ego_traj_world = np.array(fut_ego_traj_world)
        ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        pred1_len = min(pred_len, 2)
        ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        ade1s_list.append(ade1s)

        pred2_len = min(pred_len, 4)
        ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        ade2s_list.append(ade2s)

        pred3_len = min(pred_len, 6)
        ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # print(ade3s_list)
        ade3s_list.append(ade3s)

        # # Compute FDE.
        # fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # # print(f"ade: {ade}, fde: {fde}")
        # fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        # fde1s_list.append(fde1s)
        # fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        # fde2s_list.append(fde2s)
        # fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[1:pred3_len+1][-1])
        # fde3s_list.append(fde3s)

Number of scenes: 150


In [146]:
np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list)

(0.8039212266243199, nan, nan)

In [148]:
np.mean(valid_fde2s_list)

1.952224195714398

In [149]:
np.mean(valid_fde3s_list)

2.879090128077545

In [150]:
len(valid_fde3s_list)

3002

In [85]:
np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list)

(0.8609545496663229, 2.061157990841048, 3.01013521200265)

In [89]:
import json
with open('/home/can/Desktop/research/LLaDA-V/data_062525/results/nuscenes_drive_data_single_image_val_inference_zs_cot_zero_shot_cot.json', 'r') as f:
    infer_temp_data = json.load(f)

In [90]:
len(infer_temp_data)

761

## VLM_ llava

In [230]:
## train shortcut
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes

import json
# with open('/home/can/Desktop/research/LLaDA-V/data_062625/nuscenes_drive_data_single_image_val_inference_lora_llava16.json', 'r') as f:
#     infer_data = json.load(f)
with open('/home/can/Desktop/research/LLaDA-V/data_070925/nuscenes_drive_data_single_image_val_inference_lora_llava16.json', 'r') as f:
    infer_data = json.load(f)
val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid=0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # Get the camera parameters of the sample.
        camera_params.append(nusc.get('calibrated_sensor', cam_front_data['calibrated_sensor_token']))

        # Advance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []


    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # idx+=1

        data_sample = infer_data[idx]
        idx += 1
        # if idx == 3000:
        #     break
        print(data_sample[0].split('[/INST]'))
        
        speed_curvature_pred = get_status(data_sample[0].split('[/INST]')[-1])
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid+=1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            speed_curvature_pred = speed_curvature_pred[:pred_len]
            # continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            continue
        fde1s_list.append(fde1s)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[:pred2_len][-1])
        fde2s_list.append(fde2s)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        fde3s_list.append(fde3s)

        fut_ego_traj_world = np.array(fut_ego_traj_world)
        ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        pred1_len = min(pred_len, 2)
        ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        ade1s_list.append(ade1s)

        pred2_len = min(pred_len, 4)
        ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        ade2s_list.append(ade2s)

        pred3_len = min(pred_len, 6)
        ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # print(ade3s_list)
        ade3s_list.append(ade3s)

        # # Compute FDE.
        # fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # # print(f"ade: {ade}, fde: {fde}")
        # fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        # fde1s_list.append(fde1s)
        # fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        # fde2s_list.append(fde2s)
        # fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[1:pred3_len+1][-1])
        # fde3s_list.append(fde3s)

Number of scenes: 150
['[INST]  \nYou are a autonomous driving labeller. You have access to a front-view camera image of a vehicle, a sequence of past speeds, a sequence of past curvatures, and a driving rationale. Each speed, curvature is represented as [v, k], where v corresponds to the speed, and k corresponds to the curvature. A positive k means the vehicle is turning left. A negative k means the vehicle is turning right. The larger the absolute value of k, the sharper the turn. A close to zero k means the vehicle is driving straight. As a driver on the road, you should follow any common sense traffic rules. You should try to stay in the middle of your lane. You should maintain necessary distance from the leading vehicle. You should observe lane markings and follow them.  Your task is to do your best to predict future speeds and curvatures for the vehicle over the next 10 timesteps given vehicle intent inferred from the image. Make a best guess if the problem is too difficult for y

In [231]:
np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list)

(0.9097300460589918, 2.4993616987660348, 3.43663644795862)

In [232]:
invalid

1668

# Qwenvl2

In [180]:
## train shortcut
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes

import json
with open('/home/can/Desktop/research/LLaDA-V/data_062625/nuscenes_drive_data_single_image_val_inference_lora_qwen2vl.json', 'r') as f:
    infer_data = json.load(f)

val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid=0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # Get the camera parameters of the sample.
        camera_params.append(nusc.get('calibrated_sensor', cam_front_data['calibrated_sensor_token']))

        # Advance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []


    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # idx+=1

        data_sample = infer_data[idx]
        idx += 1
        # if idx == 3000:
        #     break

        
        speed_curvature_pred = get_status(data_sample[0][0])
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid+=1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            speed_curvature_pred = speed_curvature_pred[:pred_len]
            # continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            invalid+=1
            continue
        fde1s_list.append(fde1s)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        fde2s_list.append(fde2s)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        fde3s_list.append(fde3s)

        fut_ego_traj_world = np.array(fut_ego_traj_world)
        ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        pred1_len = min(pred_len, 2)
        ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        ade1s_list.append(ade1s)

        pred2_len = min(pred_len, 4)
        ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        ade2s_list.append(ade2s)

        pred3_len = min(pred_len, 6)
        ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # print(ade3s_list)
        ade3s_list.append(ade3s)

        # # Compute FDE.
        # fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # # print(f"ade: {ade}, fde: {fde}")
        # fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        # fde1s_list.append(fde1s)
        # fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        # fde2s_list.append(fde2s)
        # fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[1:pred3_len+1][-1])
        # fde3s_list.append(fde3s)

Number of scenes: 150


In [182]:
np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list)

(1.2744873453281387, 2.8521381310745273, 3.8737947367737924)

In [181]:
invalid

1

# llama32

In [183]:
## train shortcut
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes

import json
# with open('/home/can/Desktop/research/LLaDA-V/data_062625/nuscenes_drive_data_single_image_val_inference_lora_llama32.json', 'r') as f:
with open ('/home/can/Desktop/research/LLaDA-V/data_062525/results/nuscenes_drive_data_single_image_val_inference_lora_062625_llama32.json','r') as f:
    infer_data = json.load(f)

val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid = 0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # Get the camera parameters of the sample.
        camera_params.append(nusc.get('calibrated_sensor', cam_front_data['calibrated_sensor_token']))

        # Advance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []


    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # idx+=1

        data_sample = infer_data[idx]
        idx += 1
        # if idx == 3000:
        #     break

        
        speed_curvature_pred = get_status(data_sample[0].split('\n')[-1])
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid +=1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            speed_curvature_pred = speed_curvature_pred[:pred_len]
            # continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            invalid += 1
            continue
        fde1s_list.append(fde1s)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[:pred2_len][-1])
        fde2s_list.append(fde2s)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        fde3s_list.append(fde3s)

        fut_ego_traj_world = np.array(fut_ego_traj_world)
        ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        pred1_len = min(pred_len, 2)
        ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        ade1s_list.append(ade1s)

        pred2_len = min(pred_len, 4)
        ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        ade2s_list.append(ade2s)

        pred3_len = min(pred_len, 6)
        ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # print(ade3s_list)
        ade3s_list.append(ade3s)

        # # Compute FDE.
        # fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # # print(f"ade: {ade}, fde: {fde}")
        # fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        # fde1s_list.append(fde1s)
        # fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        # fde2s_list.append(fde2s)
        # fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[1:pred3_len+1][-1])
        # fde3s_list.append(fde3s)

Number of scenes: 150


In [185]:
np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list)

(0.8007789136619881, 2.3100068998217074, 3.096873512961562)

In [186]:
np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list)

(0.8007789136619881, 2.3100068998217074, 3.096873512961562)

In [184]:
invalid

2

## Step = 96

In [113]:
## train shortcut
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes

import json
with open('/home/can/Desktop/research/LLaDA-V/data_070525/nuscenes_drive_data_single_image_val_inference_sc_single_image_train_lora_sc_time_step_96.json', 'r') as f:
    infer_data = json.load(f)

val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid = 0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # nuscenes_drive_data_single_image_val_inference_lora_single_image_train_loraAdvance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []


    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # idx+=1

        data_sample = infer_data[idx]
        idx += 1
        # if idx == 3000:
        #     break

        
        speed_curvature_pred = get_status(data_sample[0][0])
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid += 1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            speed_curvature_pred = speed_curvature_pred[:pred_len]
            # continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        pred1_len = min(pred_len, 2)
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            continue
        fde1s_list.append(fde1s)
        pred2_len = min(pred_len, 4)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        fde2s_list.append(fde2s)
        valid_fde2s_list = [x for x in fde2s_list if isinstance(x, (int, float)) and not math.isnan(x)]


        pred3_len = min(pred_len, 6)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        fde3s_list.append(fde3s)
        valid_fde3s_list = [x for x in fde3s_list if isinstance(x, (int, float)) and not math.isnan(x)]
        # import math
        # for i, x in enumerate(fde3s_list):
        #     if isinstance(x, (int, float)) and not math.isnan(x):
        #         continue
        #     else:
        #         print(f"Invalid number found: {i,x}")

        fut_ego_traj_world = np.array(fut_ego_traj_world)
        # ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        # pred1_len = min(pred_len, 2)
        # ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        # ade1s_list.append(ade1s)

        # pred2_len = min(pred_len, 4)
        # ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        # ade2s_list.append(ade2s)

        # pred3_len = min(pred_len, 6)
        # ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # # print(ade3s_list)
        # ade3s_list.append(ade3s)

print(np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list),(np.mean(fde1s_list) + np.mean(fde2s_list)+ np.mean(fde3s_list))/3)

FileNotFoundError: [Errno 2] No such file or directory: '/home/can/Desktop/research/LLaDA-V/data_070525/nuscenes_drive_data_single_image_val_inference_sc_single_image_train_lora_sc_time_step_96.json'

In [107]:
fde3s_list

[0.10163930729877767,
 1.4143985187937334,
 1.765632986159155,
 1.6630103761675563,
 0.9964052537054523,
 0.2507612660476547,
 1.037427383549588,
 1.2052799651886863,
 2.1143167450739764,
 1.8358907690201927,
 0.9249119963792939,
 0.1148983680583466,
 1.622022889111952,
 0.8189467623857737,
 0.7379807045216388,
 0.4714853593446109,
 2.033976387669117,
 3.2930409221640904,
 5.037163691407689,
 5.994247590224113,
 4.433764944550935,
 2.2020196691454275,
 3.2468494116758952,
 3.0804241281441787,
 2.8575497869510498,
 3.0650379783358788,
 3.538774624386993,
 3.8172627357546003,
 3.539510611602909,
 4.4209581575095145,
 5.377463374104445,
 4.347047025292427,
 4.844780233283834,
 4.390285961257838,
 4.456979864631558,
 3.9906318237043914,
 3.317156986714105,
 2.6587387137428946,
 1.9823860421101334,
 0.8127741962630503,
 2.311590774950471,
 2.3818606179496573,
 1.5890970977121213,
 1.2946808760444677,
 3.586651121805527,
 1.9767069209938724,
 2.5712635352059436,
 1.5602660272007345,
 2.51387

In [98]:
len(valid_fde2s_list)

2993

In [101]:
len(valid_fde3s_list)

2993

In [97]:
print(np.mean(valid_fde2s_list))

1.9398157946246755


In [106]:
print(np.mean(valid_fde3s_list))

4.802040164278975


In [80]:
invalid

24

# 64

In [89]:
## train shortcut
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes

import json
with open('/home/can/Desktop/research/LLaDA-V/data_070525/nuscenes_drive_data_single_image_val_inference_sc_single_image_train_lora_sc_time_step_64.json', 'r') as f:
    infer_data = json.load(f)

val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid = 0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # nuscenes_drive_data_single_image_val_inference_lora_single_image_train_loraAdvance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []


    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # idx+=1

        data_sample = infer_data[idx]
        idx += 1
        # if idx == 3000:
        #     break

        
        speed_curvature_pred = get_status(data_sample[0][0])
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid += 1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            speed_curvature_pred = speed_curvature_pred[:pred_len]
            # print(speed_curvature_pred)
            # invalid += 1
            # continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        pred1_len = min(pred_len, 2)
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            continue
        fde1s_list.append(fde1s)
        pred2_len = min(pred_len, 4)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        fde2s_list.append(fde2s)
        pred3_len = min(pred_len, 6)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        fde3s_list.append(fde3s)

        fut_ego_traj_world = np.array(fut_ego_traj_world)
        ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        # pred1_len = min(pred_len, 2)
        # ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        # ade1s_list.append(ade1s)

        # pred2_len = min(pred_len, 4)
        # ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        # ade2s_list.append(ade2s)

        # pred3_len = min(pred_len, 6)
        # ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # # print(ade3s_list)
        # ade3s_list.append(ade3s)

print(np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list),(np.mean(fde1s_list) + np.mean(fde2s_list)+ np.mean(fde3s_list))/3)

Number of scenes: 150
0.8064061537023495 1.9327777395979624 2.8694010685014906 1.8695283206006008


In [82]:
invalid

49

# 32

In [176]:
## train shortcut
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes

import json
with open('/home/can/Desktop/research/LLaDA-V/data_070525/nuscenes_drive_data_single_image_val_inference_sc_single_image_train_lora_sc_time_step_32.json', 'r') as f:
    infer_data = json.load(f)

val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid = 0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # nuscenes_drive_data_single_image_val_inference_lora_single_image_train_loraAdvance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []


    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # idx+=1

        data_sample = infer_data[idx]
        idx += 1
        # if idx == 3000:
        #     break

        
        speed_curvature_pred = get_status(data_sample[0][0])
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid += 1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            speed_curvature_pred = speed_curvature_pred[:pred_len]
            # invalid += 1
            # continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        pred1_len = min(pred_len, 2)
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            invalid +=1
            continue
        # if fde2s > 5:
        #     continue
        fde1s_list.append(fde1s)
        pred2_len = min(pred_len, 4)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        if fde2s > 100:
            invalid+=1
            continue
        fde2s_list.append(fde2s)
        pred3_len = min(pred_len, 6)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        if fde3s > 100:
            invalid +=1
            continue
        valid_fde3s_list = [x for x in fde3s_list if isinstance(x, (int, float)) and not math.isnan(x)]
        fde3s_list.append(fde3s)

        # fut_ego_traj_world = np.array(fut_ego_traj_world)
        # ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        # pred1_len = min(pred_len, 2)
        # ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        # ade1s_list.append(ade1s)

        # pred2_len = min(pred_len, 4)
        # ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        # ade2s_list.append(ade2s)

        # pred3_len = min(pred_len, 6)
        # ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # # print(ade3s_list)
        # ade3s_list.append(ade3s)

print(np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list),(np.mean(fde1s_list) + np.mean(fde2s_list)+ np.mean(fde3s_list))/3)

Number of scenes: 150
0.8134562203754387 1.955677623860533 nan nan


In [177]:
np.mean(valid_fde3s_list)

3.0895351617503204

In [178]:
len(valid_fde3s_list)

2882

In [179]:
invalid

135

# 16

In [191]:
## train shortcut
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes

import json
with open('/home/can/Desktop/research/LLaDA-V/data_070525/nuscenes_drive_data_single_image_val_inference_sc_single_image_train_lora_sc_time_step_16.json', 'r') as f:
    infer_data = json.load(f)

val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid = 0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # nuscenes_drive_data_single_image_val_inference_lora_single_image_train_loraAdvance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []


    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # idx+=1

        data_sample = infer_data[idx]
        idx += 1
        # if idx == 3000:
        #     break

        
        speed_curvature_pred = get_status(data_sample[0][0])
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid+=1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            speed_curvature_pred = speed_curvature_pred[:pred_len]
            # continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        pred1_len = min(pred_len, 2)
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            invalid +=1
            continue
        fde1s_list.append(fde1s)
        pred2_len = min(pred_len, 4)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        fde2s_list.append(fde2s)
        pred3_len = min(pred_len, 6)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        if fde3s > 100:
            invalid +=1
            continue
        fde3s_list.append(fde3s)
        valid_fde3s_list = [x for x in fde3s_list if isinstance(x, (int, float)) and not math.isnan(x)]

        fut_ego_traj_world = np.array(fut_ego_traj_world)
        # ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        # pred1_len = min(pred_len, 2)
        # ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        # ade1s_list.append(ade1s)

        # pred2_len = min(pred_len, 4)
        # ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        # ade2s_list.append(ade2s)

        # pred3_len = min(pred_len, 6)
        # ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # # print(ade3s_list)
        # ade3s_list.append(ade3s)

print(np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list),(np.mean(fde1s_list) + np.mean(fde2s_list)+ np.mean(fde3s_list))/3)

Number of scenes: 150


/home/can/Desktop/research/LLaDA-V/utils.py:283: RuntimeWarning: invalid value encountered in multiply
  theta = cumulative_trapezoid(curvatures * velocities_norm, t, initial=0)


0.8400274075869308 1.9686855943276123 nan nan


In [192]:
np.mean(valid_fde3s_list)

3.2386113552125506

In [193]:
invalid

942

# SC + threshold 0.5

In [202]:
## train shortcut
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes

import json
with open('/home/can/Desktop/research/LLaDA-V/data_070925/nuscenes_drive_data_single_image_val_inference_sc_single_image_train_lora_sc_time_step_32_threshold_0.5.json', 'r') as f:
    infer_data = json.load(f)

val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid = 0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # nuscenes_drive_data_single_image_val_inference_lora_single_image_train_loraAdvance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []


    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # print(fut_spd_cur_str)
        # idx+=1

        data_sample = infer_data[idx]
        # print(data_sample)
        idx += 1
        # if idx == 3000:
        #     break

        
        speed_curvature_pred = get_status(data_sample[0][0])
        # print(speed_curvature_pred)
        # if speed_curvature_pred == []:
        #     continue
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid+=1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            speed_curvature_pred = speed_curvature_pred[:pred_len]
            # invalid+=1
            # continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        # print(pred)
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        pred1_len = min(pred_len, 2)
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            invalid+=1
            continue
        fde1s_list.append(fde1s)
        pred2_len = min(pred_len, 4)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        if fde2s > 100:
            invalid+=1
            continue
        fde2s_list.append(fde2s)
        # valid_fde2s_list = [x for x in fde2s_list if isinstance(x, (int, float)) and not math.isnan(x)]
        pred3_len = min(pred_len, 6)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        if fde3s > 100:
            invalid+=1
            continue
        fde3s_list.append(fde3s)
        
        fut_ego_traj_world = np.array(fut_ego_traj_world)
        # ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        # pred1_len = min(pred_len, 2)
        # ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        # ade1s_list.append(ade1s)

        # pred2_len = min(pred_len, 4)
        # ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        # ade2s_list.append(ade2s)

        # pred3_len = min(pred_len, 6)
        # ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # # print(ade3s_list)
        # ade3s_list.append(ade3s)
valid_fde3s_list = [x for x in fde3s_list if isinstance(x, (int, float)) and not math.isnan(x)]
valid_fde2s_list = [x for x in fde2s_list if isinstance(x, (int, float)) and not math.isnan(x)]
print(np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list),(np.mean(fde1s_list) + np.mean(fde2s_list)+ np.mean(fde3s_list))/3)

Number of scenes: 150
0.9651434068168648 2.407740102473603 nan nan


In [233]:
import re

def analyze_coordinate_string(s: str):
    """
    分析一个坐标对字符串，检查其是否符合预定义的“正常”格式。

    一个“正常”的字符串需要：
    1. 包含正好 10 个由 '[num, num]' 格式组成的数对。np.mean(valid_fde3s_list)
    2. 数对内的数字可以解析为浮点数。
    3. 在数对之外没有多余的字符（除了逗号、空格和可选的外部列表方括号）。

    Args:
        s: 要分析的字符串。

    Returns:
        一个元组 (is_valid, reason)，其中 is_valid 是布尔值，
        reason 是描述验证结果或失败原因的字符串。
    """
    s = s.strip()

    # 1. 使用正则表达式查找所有 `[... , ...]` 格式的数对。
    # 这个逻辑是新方法的核心，它避免了破坏原始字符串。
    try:
        # 查找数对内容，用于后续解析
        pairs_content = re.findall(r'\[([^\[\]]+)\]', s)
        # 查找完整数对（包括括号），用于下面的结构检查
        full_pairs = re.findall(r'\[[^\[\]]+\]', s)
    except Exception:
        return False, "结构异常：内部方括号不匹配或格式混乱。"

    # 2. 检查在数对之外是否有非法的残余字符。
    # 我们通过从原始字符串中移除所有找到的数对，然后检查剩下的内容。
    temp_s = s
    for pair_str in full_pairs:
        # 使用 replace 并指定 count=1，以正确处理可能重复的数对
        temp_s = temp_s.replace(pair_str, '', 1)
    
    # 移除所有合法的分隔符（空格和逗号）
    remaining_chars = re.sub(r'[\s,]', '', temp_s)
    
    # 如果原字符串是 `[[...], ...]` 格式，那么 `remaining_chars` 此时会是 "[]"
    # 我们认为这也是合法的结构，所以将其视为空。
    if remaining_chars == '[]':
        remaining_chars = ''

    if remaining_chars:
        return False, f"结构异常：在数对之外发现了多余的字符: '{remaining_chars}'"

    # 3. 检查数对的数量
    if len(pairs_content) != 10:
        return False, f"数量异常：发现了 {len(pairs_content)} 个数对，但需要正好 10 个。"

    # 4. 逐个验证每个数对的内部格式
    for i, pair_str in enumerate(pairs_content):
        parts = pair_str.split(',')
        if len(parts) != 2:
            return False, f"数对 #{i+1} 格式异常：需要2个数字，但发现了 {len(parts)} 个部分。内容: '[{pair_str}]'"
        
        try:
            # 尝试将每个部分转换为浮点数
            float(parts[0].strip())
            float(parts[1].strip())
        except ValueError:
            return False, f"数值异常：数对 #{i+1} 中包含无法解析为数字的内容。内容: '[{pair_str}]'"
            
    return True, "格式正常"


def fix_coordinate_string(s: str):
    """
    尝试将一个格式异常的坐标字符串修复为最接近的正确格式。

    ### 修复方案设计 ###

    我们的目标是：对于一个格式错误的字符串，生成一个结构最接近它的、
    并且完全符合“正常格式”的字符串。

    核心思路是“提取、清理、重建”：

    1.  **提取 (Extract)**：
        不论原始字符串多么混乱，首先用正则表达式尽可能地提取出所有
        看起来像数对 `[...]` 的部分。

    2.  **清理 (Clean)**：
        对提取出的每个部分进行严格验证。如果一个部分不包含两个由逗号
        分隔的、可以转换为数字的值，就直接丢弃它。这比猜测如何修正
        它更安全。所有通过验证的数对，我们将其保存为一个干净的 Python
        列表，例如 [[0.82, 0.0], [0.82, 0.01], ...]。

    3.  **重建 (Rebuild)**：
        -   **处理数量问题**：检查清理后的数对列表。
            -   如果数量超过 10 个，则截断，只保留前 10 个。
            -   如果数量少于 10 个，则用默认的 `[0.0, 0.0]` 来补足，
                直到正好有 10 个。
        -   **格式化输出**：将最终的 10 个数对格式化成一个标准的、
            干净的字符串。

    Args:
        s: 格式异常的字符串。

    Returns:
        一个元组 (fixed_string, reason)，包含修复后的字符串和修复操作的描述。
    """
    # 1. 提取所有可能的数对内容
    try:
        pairs_content = re.findall(r'\[([^\[\]]+)\]', s)
    except Exception:
        return s, "修复失败：正则表达式无法解析该字符串。"

    # 2. 清理和验证，只保留有效的数对
    valid_pairs = []
    for content in pairs_content:
        parts = [p.strip() for p in content.split(',')]
        if len(parts) != 2:
            continue
        try:
            num1 = float(parts[0])
            num2 = float(parts[1])
            valid_pairs.append([num1, num2])
        except ValueError:
            continue

    # 3. 重建 - 处理数量问题
    num_valid = len(valid_pairs)
    fix_reason = ""
    if num_valid > 10:
        fixed_pairs = valid_pairs[:10]
        fix_reason = f"修复操作：保留了前 10 个有效数对 (原先有 {num_valid} 个)。"
    elif num_valid < 10:
        num_to_add = 10 - num_valid
        fixed_pairs = valid_pairs + [[0.0, 0.0]] * num_to_add
        fix_reason = f"修复操作：补齐了 {num_to_add} 个 '[0.0, 0.0]' 数对 (原先仅 {num_valid} 个有效)。"
    else:
        fixed_pairs = valid_pairs
        fix_reason = "修复操作：清理了无效字符和格式，并重新构建了字符串。"

    # 4. 重建 - 格式化输出
    # 使用 f-string 和格式说明符 `.2f` 保证所有数字都有两位小数，非常整洁
    reconstructed_pairs = [f"[{p[0]:.2f}, {p[1]:.2f}]" for p in fixed_pairs]
    final_string = f"[{', '.join(reconstructed_pairs)}]"
    
    return final_string, fix_reason

In [259]:
def fix_coordinate_string(s: str):
    """
    尝试将一个格式异常的坐标字符串修复为最接近的正确格式。
    此函数采用基于位置的插值修复策略。

    ### 修复方案设计：基于位置的插值修复 ###

    核心思路是根据数对在原始字符串中的位置来智能识别空缺并填补。

    1.  **精准提取与定位 (Extract & Locate)**
        使用 `re.finditer` 获取每个有效数对的内容和其在原始字符串中的起止位置。

    2.  **位置映射 (Position Mapping)**
        根据数对在原始字符串中的相对位置，将其映射到10个逻辑位置中的合适位置。
        映射策略：将字符串分为10个等长区间，每个数对根据其中心位置确定应该属于哪个区间。

    3.  **两阶段冲突解决 (Two-Phase Conflict Resolution)**
        - 第一阶段：只填充没有冲突的位置，将冲突的数对暂存。
        - 第二阶段：为冲突的数对寻找剩余的可用空位。

    4.  **插值填补 (Interpolate & Fill)**
        -   **双侧邻居存在**：执行线性插值。
        -   **单侧邻居存在**：直接复用邻居的值。
        -   **无邻居**：使用 `[0.0, 0.0]` 填充。

    5.  **重建字符串 (Rebuild)**
        将10个修复好的数对格式化成一个统一、干净的标准字符串。
    """
    # 1. 精准提取与定位
    found_pairs = []
    for match in re.finditer(r'\[([^\[\]]+)\]', s):
        content = match.group(1)
        parts = [p.strip() for p in content.split(',')]
        if len(parts) == 2:
            try:
                num1 = float(parts[0])
                num2 = float(parts[1])
                # 计算数对的中心位置
                center_pos = (match.span()[0] + match.span()[1]) / 2
                found_pairs.append({
                    'value': [num1, num2], 
                    'span': match.span(),
                    'center': center_pos
                })
            except ValueError:
                continue
    
    # 如果找不到任何有效数对，返回一个默认字符串
    if not found_pairs:
        default_pair = "[0.00, 0.00]"
        return f"[{', '.join([default_pair] * 10)}]", "修复操作：未找到任何有效数对，已生成默认字符串。"
    
    # 如果数对数量够多，直接截取前10个并重建
    if len(found_pairs) >= 10:
        final_pairs_values = [p['value'] for p in found_pairs[:10]]
        fix_reason = f"修复操作：保留了前 10 个有效数对 (原先有 {len(found_pairs)} 个)。"
    else:
        # 2. 位置映射：根据数对在原始字符串中的位置，映射到10个逻辑位置
        final_pairs_values = [None] * 10
        string_length = len(s)
        
        # 如果字符串太短，使用简单的均匀分布作为后备方案
        if string_length < 10:
            if len(found_pairs) == 1:
                final_pairs_values[4] = found_pairs[0]['value']
            else:
                step = 9.0 / (len(found_pairs) - 1)
                for i, pair in enumerate(found_pairs):
                    pos = int(round(i * step))
                    while pos < 10 and final_pairs_values[pos] is not None:
                        pos += 1
                    if pos < 10:
                        final_pairs_values[pos] = pair['value']
        else:
            # 将字符串分为10个等长区间
            interval_length = string_length / 10.0
            
            # 3. 两阶段冲突解决
            
            # 第一阶段：计算所有数对的目标位置，并识别冲突
            position_mapping = {}  # logical_pos -> [pairs_list]
            
            for pair in found_pairs:
                # 根据数对的中心位置确定它应该属于哪个区间
                logical_pos = int(round(pair['center'] / interval_length))
                logical_pos = min(max(logical_pos, 0), 9)  # 保证范围在 [0,9]
                
                if logical_pos not in position_mapping:
                    position_mapping[logical_pos] = []
                position_mapping[logical_pos].append(pair)
            
            # 第一阶段：只填充没有冲突的位置
            conflicted_pairs = []
            
            for logical_pos, pairs_list in position_mapping.items():
                if len(pairs_list) == 1:
                    # 没有冲突，直接填充
                    final_pairs_values[logical_pos] = pairs_list[0]['value']
                else:
                    # 有冲突，保留第一个，其余的放入冲突列表
                    final_pairs_values[logical_pos] = pairs_list[0]['value']
                    conflicted_pairs.extend(pairs_list[1:])
            
            # 第二阶段：为冲突的数对寻找剩余的可用空位
            for pair in conflicted_pairs:
                # 寻找任何可用的空位
                placed = False
                
                # 优先寻找距离原始目标位置最近的空位
                original_pos = int(round(pair['center'] / interval_length))
                original_pos = min(max(original_pos, 0), 9)  # 保证范围在 [0,9]
                
                # 从原始位置开始，向两边扩展搜索
                for offset in range(10):
                    # 向右搜索
                    if original_pos + offset < 10 and final_pairs_values[original_pos + offset] is None:
                        final_pairs_values[original_pos + offset] = pair['value']
                        placed = True
                        break
                    # 向左搜索（offset > 0 时）
                    if offset > 0 and original_pos - offset >= 0 and final_pairs_values[original_pos - offset] is None:
                        final_pairs_values[original_pos - offset] = pair['value']
                        placed = True
                        break
                
                # 如果仍然找不到空位（理论上不应该发生，因为我们最多只有9个数对要放入10个位置）
                if not placed:
                    # 这种情况下，我们可能需要重新评估算法，但作为最后的保障
                    print(f"警告：无法为数对 {pair['value']} 找到合适的位置")
        
        # 4. 插值填补空缺位置
        num_missing = sum(1 for x in final_pairs_values if x is None)
        
        for i in range(10):
            if final_pairs_values[i] is None:
                # 寻找最近的前后邻居
                prev_value = None
                prev_idx = -1
                for j in range(i-1, -1, -1):
                    if final_pairs_values[j] is not None:
                        prev_value = final_pairs_values[j]
                        prev_idx = j
                        break
                
                next_value = None
                next_idx = -1
                for j in range(i+1, 10):
                    if final_pairs_values[j] is not None:
                        next_value = final_pairs_values[j]
                        next_idx = j
                        break
                
                if prev_value and next_value:
                    # 线性插值
                    ratio = (i - prev_idx) / (next_idx - prev_idx)
                    x = prev_value[0] + ratio * (next_value[0] - prev_value[0])
                    y = prev_value[1] + ratio * (next_value[1] - prev_value[1])
                    final_pairs_values[i] = [x, y]
                elif prev_value:
                    # 只有前邻居，复用
                    final_pairs_values[i] = prev_value[:]
                elif next_value:
                    # 只有后邻居，复用
                    final_pairs_values[i] = next_value[:]
                else:
                    # 使用默认值
                    final_pairs_values[i] = [0.0, 0.0]
        
        conflict_count = len(conflicted_pairs)
        if conflict_count > 0:
            fix_reason = f"修复操作：基于位置映射和插值的方式补齐了 {num_missing} 个数对（解决了 {conflict_count} 个位置冲突）。"
        else:
            fix_reason = f"修复操作：基于位置映射和插值的方式补齐了 {num_missing} 个数对。"

    # 5. 重建字符串
    reconstructed_pairs = [f"[{p[0]:.2f}, {p[1]:.2f}]" for p in final_pairs_values]
    final_string = f"[{', '.join(reconstructed_pairs)}]"
    
    return final_string, fix_reason


In [260]:
## train shortcut with fix
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes

import json
with open('/home/can/Desktop/research/LLaDA-V/data_070925/nuscenes_drive_data_single_image_val_inference_sc_single_image_train_lora_sc_time_step_32_threshold_0.5.json', 'r') as f:
    infer_data = json.load(f)

# extracted_strings = re.findall(r'\"(.*?)\"', infer_data, re.DOTALL)

val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid = 0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of thnp.mean(valid_fde3s_list)e sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # nuscenes_drive_data_single_image_val_inference_lora_single_image_train_loraAdvance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []


    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # print(fut_spd_cur_str)
        # idx+=1

        data_sample = infer_data[idx]
        is_valid, reason = analyze_coordinate_string(data_sample[0][0])
        # print('raw',get_status(data_sample[0][0]))
        if not is_valid:
            # print('raw',type(data_sample[0][0]))
            # continue
            # print(type())
            speed_curvature_pred = get_status(fix_coordinate_string(data_sample[0][0])[0])
            # print(type(speed_curvature_pred))
        else:
            speed_curvature_pred = get_status(data_sample[0][0])
            # print(f"Valid: {type(speed_curvature_pred)}")




        # print(data_sample)
        idx += 1

        # speed_curvature_pred = get_status(data_sample[0][0])
        # print(speed_curvature_pred)
        # if speed_curvature_pred == []:
        #     continue
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid+=1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            speed_curvature_pred = speed_curvature_pred[:pred_len]
            # invalid+=1
            # continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        # print(pred)
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)

        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        pred1_len = min(pred_len, 2)
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            invalid+=1
            continue
        fde1s_list.append(fde1s)
        pred2_len = min(pred_len, 4)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        if fde2s > 100:
            invalid+=1
            continue
        fde2s_list.append(fde2s)
        # valid_fde2s_list = [x for x in fde2s_list if isinstance(x, (int, float)) and not math.isnan(x)]
        pred3_len = min(pred_len, 6)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        if fde3s > 100:
            invalid+=1
            continue
        fde3s_list.append(fde3s)
        
        fut_ego_traj_world = np.array(fut_ego_traj_world)

valid_fde3s_list = [x for x in fde3s_list if isinstance(x, (int, float)) and not math.isnan(x)]
valid_fde2s_list = [x for x in fde2s_list if isinstance(x, (int, float)) and not math.isnan(x)]
print(np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list),(np.mean(fde1s_list) + np.mean(fde2s_list)+ np.mean(fde3s_list))/3)

Number of scenes: 150
0.9468720331731346 2.3718376766145304 nan nan


In [261]:
np.mean(valid_fde2s_list),np.mean(valid_fde3s_list),invalid

(2.3718376766145304, 4.863559654440921, 13)

In [204]:
np.mean(valid_fde2s_list)

2.407740102473603

In [205]:
np.mean(valid_fde3s_list)

4.983602990372428

In [203]:
invalid

1032

# threshold 0.7

In [65]:
## train shortcut
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes

import json
with open('/home/can/Desktop/research/LLaDA-V/data_070825/nuscenes_drive_data_single_image_val_inference_sc_single_image_train_lora_sc_time_step_32_threshold_0.7.json', 'r') as f:
    infer_data = json.load(f)

val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid = 0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # nuscenes_drive_data_single_image_val_inference_lora_single_image_train_loraAdvance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []


    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # idx+=1

        data_sample = infer_data[idx]
        idx += 1
        # if idx == 3000:
        #     break

        
        speed_curvature_pred = get_status(data_sample[0][0])
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid+=1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        pred1_len = min(pred_len, 2)
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            # invalid +=1
            continue
        fde1s_list.append(fde1s)
        pred2_len = min(pred_len, 4)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        fde2s_list.append(fde2s)
        pred3_len = min(pred_len, 6)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        fde3s_list.append(fde3s)

        fut_ego_traj_world = np.array(fut_ego_traj_world)
        ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        pred1_len = min(pred_len, 2)
        ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        ade1s_list.append(ade1s)

        pred2_len = min(pred_len, 4)
        ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        ade2s_list.append(ade2s)

        pred3_len = min(pred_len, 6)
        ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # print(ade3s_list)
        ade3s_list.append(ade3s)

print(np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list),(np.mean(fde1s_list) + np.mean(fde2s_list)+ np.mean(fde3s_list))/3)

Number of scenes: 150
0.9031726351117925 2.114305677046473 3.4775575550301245 2.1650119557294634


In [66]:
invalid

569

# threshold 0.9

In [225]:
## train shortcut
import os
import numpy as np
from utils import EstimateCurvatureFromTrajectory, IntegrateCurvatureForPoints, OverlayTrajectory, WriteImageSequenceToVideo
from math import atan2
OBS_LEN = 10
FUT_LEN = 10
TTL_LEN = OBS_LEN + FUT_LEN
data_nuscenes = []
idx = 0
from nuscenes.utils.splits import create_splits_scenes

import json
with open('/home/can/Desktop/research/LLaDA-V/data_070925/nuscenes_drive_data_single_image_val_inference_sc_single_image_train_lora_sc_time_step_32_threshold_0.9.json', 'r') as f:
    infer_data = json.load(f)

val_scenes = create_splits_scenes()['val']

print(f"Number of scenes: {len(val_scenes)}")
# from nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-trainval', dataroot='/media/data/nuscenes/full', verbose=True)
# scenes = nusc.scene
ade1s_list = []
ade2s_list = []
ade3s_list = []
fde1s_list = []
fde2s_list = []
fde3s_list = []
invalid = 0
for scene in scenes:
    if scene['name'] not in val_scenes:
        continue
    token = scene['token']
    first_sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']
    name = scene['name']
    description = scene['description']

    # if not name in ["scene-0103", "scene-1077"]:
    #     continue

    # Get all image and pose in this scene
    front_camera_images = []
    ego_poses = []
    camera_params = []
    curr_sample_token = first_sample_token
    while True:
        sample = nusc.get('sample', curr_sample_token)

        # Get the front camera image of the sample.
        cam_front_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        # nusc.render_sample_data(cam_front_data['token'])



        front_camera_images.append(os.path.join(nusc.dataroot, cam_front_data['filename']).replace('/media','/scratch/gilbreth/cancui'))

        # Get the ego pose of the sample.
        pose = nusc.get('ego_pose', cam_front_data['ego_pose_token'])
        ego_poses.append(pose)

        # nuscenes_drive_data_single_image_val_inference_lora_single_image_train_loraAdvance the pointer.
        if curr_sample_token == last_sample_token:
            break
        curr_sample_token = sample['next']

    scene_length = len(front_camera_images)
    # print(f"Scene {name} has {scene_length} frames")

    if scene_length < TTL_LEN:
        print(f"Scene {name} has less than {TTL_LEN} frames, skipping...")
        continue

    # Get the velocities of the ego vehicle.
    ego_poses_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]
    ego_poses_world = np.array(ego_poses_world)
    # plt.plot(ego_poses_world[:, 0], ego_poses_world[:, 1], 'r-', label='GT')

    ego_velocities = np.zeros_like(ego_poses_world)
    ego_velocities[1:] = ego_poses_world[1:] - ego_poses_world[:-1]
    ego_velocities[0] = ego_velocities[1]

    # Get the curvature of the ego vehicle.
    ego_curvatures = EstimateCurvatureFromTrajectory(ego_poses_world)
    ego_velocities_norm = np.linalg.norm(ego_velocities, axis=1)
    estimated_points = IntegrateCurvatureForPoints(ego_curvatures, ego_velocities_norm, ego_poses_world[0],
                                                    atan2(ego_velocities[0][1], ego_velocities[0][0]), scene_length)


    ego_traj_world = [ego_poses[t]['translation'][:3] for t in range(scene_length)]

    prev_intent = None
    cam_images_sequence = []


    for i in range(scene_length - TTL_LEN):
        # Get the raw image data.
        # utils.PlotBase64Image(front_camera_images[0])
        obs_images = front_camera_images[i:i+OBS_LEN]
        obs_ego_poses = ego_poses[i:i+OBS_LEN]
        obs_camera_params = camera_params[i:i+OBS_LEN]
        obs_ego_traj_world = ego_traj_world[i:i+OBS_LEN]
        fut_ego_traj_world = ego_traj_world[i+OBS_LEN:i+TTL_LEN]
        obs_ego_velocities = ego_velocities[i:i+OBS_LEN]
        obs_ego_velocities_norm = np.linalg.norm(obs_ego_velocities, axis=1)
        obs_ego_curvatures = ego_curvatures[i:i+OBS_LEN]
        obs_speed_curvature_str = convert_to_speed_curvature_template(obs_ego_velocities_norm, obs_ego_curvatures)
        fut_ego_velocities = ego_velocities[i+OBS_LEN:i+TTL_LEN]
        fut_ego_curvatures = ego_curvatures[i+OBS_LEN:i+TTL_LEN]
        fut_ego_velocities_norm = np.linalg.norm(fut_ego_velocities, axis=1)
        # Get positions of the vehicle.
        obs_start_world = obs_ego_traj_world[0]
        fut_start_world = obs_ego_traj_world[-1]
        curr_image = obs_images[-1]
        fut_spd_cur_str = convert_to_speed_curvature_template(fut_ego_velocities_norm, fut_ego_curvatures)
        # idx+=1

        data_sample = infer_data[idx]
        idx += 1
        # if idx == 3000:
        #     break

        
        speed_curvature_pred = get_status(data_sample[0][0])
        # print(data_sample[0])
        speed_curvature_gt = get_status(data_sample[1])
        pred_len = min(FUT_LEN, len(speed_curvature_pred))
        if len(speed_curvature_pred) < 2 or speed_curvature_pred == []:
            invalid +=1
            continue
        if len(speed_curvature_pred) != len(speed_curvature_gt) or len(speed_curvature_gt) != pred_len:
            speed_curvature_pred = speed_curvature_pred[:pred_len]
            # invalid +=1
            # continue
        pred_curvatures = np.array(speed_curvature_pred)[:, 1] / 100
        pred_speeds = np.array(speed_curvature_pred)[:, 0]
        pred_traj = np.zeros((pred_len, 3))
        assert pred_curvatures.size == pred_speeds.size, f"Predicted curvatures length {len(pred_curvatures)} does not match speed predicted length {len(pred_speeds)}"
        pred_traj[:pred_len, :2] = IntegrateCurvatureForPoints(pred_curvatures,
                                                                   pred_speeds,
                                                                   fut_start_world,
                                                                   atan2(obs_ego_velocities[-1][1],
                                                                         obs_ego_velocities[-1][0]), pred_len)
        # check_flag = OverlayTrajectory(img, pred_traj.tolist(), obs_camera_params[-1], obs_ego_poses[-1], color=(255, 0, 0), args=args)
            

        # Compute ADE.
        # Compute FDE.
        fde = np.linalg.norm(fut_ego_traj_world[:pred_len][-1] - pred_traj[-1])
        # print(f"ade: {ade}, fde: {fde}")
        pred1_len = min(pred_len, 2)
        fde1s = np.linalg.norm(fut_ego_traj_world[:pred1_len][-1] - pred_traj[1:pred1_len+1][-1])
        if fde1s > 10:
            continue
        fde1s_list.append(fde1s)
        pred2_len = min(pred_len, 4)
        fde2s = np.linalg.norm(fut_ego_traj_world[:pred2_len][-1] - pred_traj[1:pred2_len+1][-1])
        if fde3s > 100:
            invalid +=1
            continue
        fde2s_list.append(fde2s)
        pred3_len = min(pred_len, 6)
        fde3s = np.linalg.norm(fut_ego_traj_world[:pred3_len][-1] - pred_traj[:pred3_len][-1])
        if fde3s > 100:
            invalid +=1
            continue
        fde3s_list.append(fde3s)

        fut_ego_traj_world = np.array(fut_ego_traj_world)
        # ade = np.mean(np.linalg.norm(fut_ego_traj_world[:pred_len] - pred_traj, axis=1))
        
        # pred1_len = min(pred_len, 2)
        # ade1s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred1_len] - pred_traj[1:pred1_len+1] , axis=1))
        # ade1s_list.append(ade1s)

        # pred2_len = min(pred_len, 4)
        # ade2s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred2_len] - pred_traj[1:pred2_len+1] , axis=1))
        # ade2s_list.append(ade2s)

        # pred3_len = min(pred_len, 6)
        # ade3s = np.mean(np.linalg.norm(fut_ego_traj_world[:pred3_len] - pred_traj[:pred3_len] , axis=1))
        # # print(ade3s_list)
        # ade3s_list.append(ade3s)

print(np.mean(fde1s_list), np.mean(fde2s_list), np.mean(fde3s_list),(np.mean(fde1s_list) + np.mean(fde2s_list)+ np.mean(fde3s_list))/3)

Number of scenes: 150
0.8232056418868834 1.9632554996156126 3.109934754440583 1.965465298647693


In [226]:
invalid/len(infer_data)

0.07485922490891024